In [ ]:
from pathlib import Path
import itertools

import numpy as np
import pandas as pd
from tqdm import tqdm

from response_stats.pipeline import SimulationConfig, ResponseSimulator, _force_matlab_cell_structure, _parse_save_by_language, _as_cell_of_scalars, _as_numeric_vector, _is_sequence, _as_cell_of_vectors
from response_stats.generators import PoissonSpikeGenerator

from response_stats.plot_utils import *

from scipy.ndimage import gaussian_filter

In [ ]:
baseline_frs = np.arange(0.1, 15.1, 0.2)
response_frs = np.arange(0.1, 50, 0.2)

fr_pairs = np.array([[b, r] for b in baseline_frs for r in response_frs])
n_samples = len(fr_pairs)

fr_baseline = fr_pairs[:,0]
fr_response = fr_pairs[:,1]

save_dir = Path("/media/al/darch/response_stats/datasets/test")

In [ ]:
### trials 5- 15
cfg = SimulationConfig(
    # =========================
    # General
    # =========================
    n_samples=n_samples,
    save_dir=save_dir,
    save_language="both",

    # =========================
    # Response type
    # =========================
    response_type="response",

    # =========================
    # Trial parameters
    # =========================
    trial_range=[5, 15],

    # =========================
    # Time parameters
    # =========================
    baseline_T=2.0,
    stimulus_T=2.0,
    dt=0.001,

    # =========================
    # Response shape (beta params)
    # =========================
    beta_a_range=[1, 1.5],
    beta_multiplier_range=[1.5, 2],

    # =========================
    # Response duration
    # =========================
    duration_range=[0.25, 0.7],

    # =========================
    # Response latency
    # =========================
    latency_range=[0.25, 0.41],

    induce_refractory_period=True,

    # =========================
    # Supplementary trials
    # =========================
    generate_supplementary_trials=True,
    supplementary_default_count=200,

    # =========================
    # Random seed
    # =========================
    seed=73,

    # =========================
    # Baseline firing rate
    # =========================
    baseline_threshold=False,
    baseline_scale=False,
    baseline_range=[0, 15],

    # =========================
    # Peak response firing rate
    # =========================
    response_fr_sampler="uniform",
    response_fr_method="linear_function",
    response_fr_scale=False,
    response_fr_max=60,

    # =========================
    # Gain response function
    # =========================
    gain_response_fixed=False,
    gain_response_y=False,
    gain_response_high=False,
    gain_response_e=False,

    # =========================
    # Linear response function
    # =========================
    linear_response_slope=1,
    linear_response_offset=0,


    # =========================
    # Burst Params
    # =========================
    include_bursts=True,
    burst_rate_baseline=0,
    burst_rate_response=15,
    burst_rate_baseline_scale=1,
    burst_rate_response_scale=2,
    burst_duration_lam=150,
    burst_response_time_factor=5,
    burst_alpha=1.5,
    burst_beta=3,
    burst_multiplier=10,
    
)


rng = np.random.default_rng(seed=73)

n_trials = 171

baseline_fr = 13
response_fr = 25.5

duration = 0.31099
latency = 0.335616

a = 1.153911
b = 1.945423

burst_rate_baseline = 0.5
burst_rate_factor   = 0.25
burst_rate_response = 15.

generator = PoissonSpikeGenerator(
baseline_fr=baseline_fr,
response_fr=response_fr,
latency=latency,
duration=duration,
baseline_T=cfg.baseline_T,
stimulus_T=cfg.stimulus_T,
dt=cfg.dt,
induce_refractory_period=cfg.induce_refractory_period,
rng=rng,
a=a,
b=b, 
include_bursts=cfg.include_bursts, 
        burst_rate_baseline=burst_rate_baseline, 
        burst_rate_factor=burst_rate_factor,
        burst_rate_response=burst_rate_response, 
        burst_response_time_factor=cfg.burst_response_time_factor, 
        # trial-wise burst params
        burst_duration_lam=cfg.burst_duration_lam,
        burst_alpha=cfg.burst_alpha, 
        burst_beta=cfg.burst_beta,
        burst_multiplier=cfg.burst_multiplier,
)

# generate trials
trial_activity = generator.generate(n_trials)

baseline_time = 0
stimulus_time = 4
bin_size = 25 / 1000
sigma = 1
bins = np.arange(baseline_time, stimulus_time + (bin_size), bin_size)
binned_spikes = np.array([np.histogram(e, bins=bins)[0] for e in trial_activity]) / bin_size
mean_fr = np.mean(binned_spikes, axis=0)
smooth_fr = np.mean(gaussian_filter(binned_spikes, sigma=sigma), axis=0)
bin_centers = np.convolve(bins, np.ones(2) / 2, mode="valid")

mean_fr_response = mean_fr[(bin_centers > generator.response_onset / 1000) & (bin_centers < generator.response_offset / 1000)]
smooth_fr_response = smooth_fr[(bin_centers > generator.response_onset / 1000) & (bin_centers < generator.response_offset / 1000)]

estimated_peak_baseline = round(np.mean(mean_fr[bin_centers < 2]), 4)
estimated_peak_response = max(mean_fr)

gt_response_period_mean = np.mean(generator.r_t[generator.response_onset:generator.response_offset])

b = np.array(list(itertools.chain.from_iterable([np.diff(t) for t in trial_activity])))
isis_v = sum(b < (3 / 1000)) / len(b) * 100


fig, axes = plt.subplots(2,1, figsize=(5, 10), sharex=True)

ax = axes[0]
ax.eventplot(trial_activity)
ax.vlines(2, 0, n_trials, lw=3, color="grey")
sns.despine(left=True, ax=ax)

if cfg.include_bursts:
    s = f"baseline burst rate: {generator.burst_rate_baseline}  baseline burst mult.:{generator.burst_multiplier_baseline}\nresponse burst rate: {generator.burst_rate_response}    response burst mult.:{generator.burst_multiplier_response}\nISI violations: {round(isis_v, 3)}%"
else:
    s = f"no bursts\nISI violations: {round(isis_v, 3)}%"
ax.set_title(s)

ax = axes[1]
ax.vlines(2, min(mean_fr)-0.1, max(mean_fr)+0.1, lw=3, color="grey")
ax.plot(bin_centers, mean_fr, label=f"binwise avg. FR, {bin_size} s bins")
ax.plot(bin_centers, smooth_fr, label=f"smoothed FR, Gauss, sigma = {sigma}")
ax.set_xlabel("Time [s]")
ax.set_ylabel("FR [Hz]")
#ax.set_yticks(np.linspace(baseline_fr, response_fr,5))


ax.text(0.95, 0.95, f"GT baseline FR: {baseline_fr}", color="black", transform = ax.transAxes)
ax.text(0.95, 0.9, f"avg. baseline FR: {estimated_peak_baseline}", color="tab:blue", transform = ax.transAxes)
ax.text(0.95, 0.85, f"avg. baseline FR: {round(np.mean(smooth_fr[bin_centers < 2]), 4)}", color="tab:orange", transform = ax.transAxes)

ax.text(0.95, 0.75, f"GT peak response FR: {response_fr}", color="black", transform = ax.transAxes)
ax.text(0.95, 0.7, f"peak FR: {estimated_peak_response}", color="tab:blue", transform = ax.transAxes)
ax.text(0.95, 0.65, f"peak FR: {round(max(smooth_fr), 4)}", color="tab:orange", transform = ax.transAxes)

ax.text(0.95, 0.55, f"GT avg. response FR: {round(gt_response_period_mean, 4)}", color="black", transform = ax.transAxes)
ax.text(0.95, 0.5, f"avg. response FR: {round(np.mean(mean_fr_response),4)}", color="tab:blue", transform = ax.transAxes)
ax.text(0.95, 0.45, f"avg. response FR: {round(np.mean(smooth_fr_response), 4)}", color="tab:orange", transform = ax.transAxes)

ax.legend(loc='center left', bbox_to_anchor=(.95, 0.3))
sns.despine(ax=ax)

plt.show()

In [ ]:
print("including")
n_samples = 200

collect_baseline_mean_fr = []
collect_baseline_smooth_fr = []

collect_response_peak_mean_fr = []
collect_response_peak_smooth_fr = []

collect_response_period_mean_fr = []
collect_response_period_smooth_fr = []

collect_isi_v = []

###

n_trials = 100

baseline_fr  = 5
response_fr  = 50

duration = 0.5
latency = 0.3
a =1.1
b = 2

burst_rate_baseline = 1.500993
burst_rate_factor   = 2
burst_rate_response = 17.449581

baseline_time = 0
stimulus_time = 4
bin_size = 25 / 1000
sigma = 1


n = 0
while n < n_samples:
    rng = np.random.default_rng()

    generator = PoissonSpikeGenerator(
    baseline_fr=baseline_fr,
    response_fr=response_fr,
    latency=latency,
    duration=duration,
    baseline_T=cfg.baseline_T,
    stimulus_T=cfg.stimulus_T,
    dt=cfg.dt,
    induce_refractory_period=cfg.induce_refractory_period,
    rng=rng,
    a=a,
    b=b, 
    include_bursts=cfg.include_bursts, 
    burst_rate_baseline=burst_rate_baseline, 
    burst_rate_factor=burst_rate_factor,
    burst_rate_response=burst_rate_response, 
        burst_response_time_factor=cfg.burst_response_time_factor, 
        # trial-wise burst params
        burst_duration_lam=cfg.burst_duration_lam,
        burst_alpha=cfg.burst_alpha, 
        burst_beta=cfg.burst_beta,
        burst_multiplier=cfg.burst_multiplier - baseline_fr,
)

    # generate trials
    trial_activity = generator.generate(n_trials)

    bins = np.arange(baseline_time, stimulus_time + (bin_size), bin_size)
    binned_spikes = np.array([np.histogram(e, bins=bins)[0] for e in trial_activity]) / bin_size
    mean_fr = np.mean(binned_spikes, axis=0)
    smooth_fr = np.mean(gaussian_filter(binned_spikes, sigma=sigma), axis=0)
    bin_centers = np.convolve(bins, np.ones(2) / 2, mode="valid")

    mean_fr_response = mean_fr[(bin_centers > generator.response_onset / 1000) & (bin_centers < generator.response_offset / 1000)]
    smooth_fr_response = smooth_fr[(bin_centers > generator.response_onset / 1000) & (bin_centers < generator.response_offset / 1000)]

    estimated_peak_baseline = round(np.mean(mean_fr[bin_centers < 2]), 4)
    estimated_peak_response = max(mean_fr)

    gt_response_period_mean = np.mean(generator.r_t[generator.response_onset:generator.response_offset])

    all_isis = np.array(list(itertools.chain.from_iterable([np.diff(t) for t in trial_activity])))
    isis_v = sum(all_isis < (3 / 1000)) / len(all_isis) * 100
    
    collect_baseline_mean_fr.append(estimated_peak_baseline)
    collect_baseline_smooth_fr.append(np.mean(smooth_fr[bin_centers < 2]))

    collect_response_peak_mean_fr.append(estimated_peak_response)
    collect_response_peak_smooth_fr.append(max(smooth_fr))

    collect_response_period_mean_fr.append(np.mean(mean_fr_response) - gt_response_period_mean)
    collect_response_period_smooth_fr.append(np.mean(smooth_fr_response) - gt_response_period_mean)

    collect_isi_v.append(isis_v)

    n += 1

In [ ]:
print("excluding")
n_samples = 200

collect_baseline_mean_fr_no_bursts = []
collect_baseline_smooth_fr_no_bursts = []

collect_response_peak_mean_fr_no_bursts = []
collect_response_peak_smooth_fr_no_bursts = []

collect_response_period_mean_fr_no_bursts = []
collect_response_period_smooth_fr_no_bursts = []

collect_isi_v_no_bursts = []

###

n = 0
while n < n_samples:
    rng = np.random.default_rng()

    generator = PoissonSpikeGenerator(
    baseline_fr=baseline_fr,
    response_fr=response_fr,
    latency=latency,
    duration=duration,
    baseline_T=cfg.baseline_T,
    stimulus_T=cfg.stimulus_T,
    dt=cfg.dt,
    induce_refractory_period=cfg.induce_refractory_period,
    rng=rng,
    a=a,
    b=b, 
    include_bursts=False, 
    burst_rate_baseline=burst_rate_baseline, 
    burst_rate_factor=burst_rate_factor,
    burst_rate_response=burst_rate_response, 
        burst_response_time_factor=cfg.burst_response_time_factor, 
        # trial-wise burst params
        burst_duration_lam=cfg.burst_duration_lam,
        burst_alpha=cfg.burst_alpha, 
        burst_beta=cfg.burst_beta,
        burst_multiplier=cfg.burst_multiplier - baseline_fr,
)



    # generate trials
    trial_activity = generator.generate(n_trials)

    bins = np.arange(baseline_time, stimulus_time + (bin_size), bin_size)
    binned_spikes = np.array([np.histogram(e, bins=bins)[0] for e in trial_activity]) / bin_size
    mean_fr = np.mean(binned_spikes, axis=0)
    smooth_fr = np.mean(gaussian_filter(binned_spikes, sigma=sigma), axis=0)
    bin_centers = np.convolve(bins, np.ones(2) / 2, mode="valid")

    mean_fr_response = mean_fr[(bin_centers > generator.response_onset / 1000) & (bin_centers < generator.response_offset / 1000)]
    smooth_fr_response = smooth_fr[(bin_centers > generator.response_onset / 1000) & (bin_centers < generator.response_offset / 1000)]

    estimated_peak_baseline = round(np.mean(mean_fr[bin_centers < 2]), 4)
    estimated_peak_response = max(mean_fr)

    gt_response_period_mean = np.mean(generator.r_t[generator.response_onset:generator.response_offset])

    all_isis = np.array(list(itertools.chain.from_iterable([np.diff(t) for t in trial_activity])))
    isis_v = sum(all_isis < (3 / 1000)) / len(all_isis) * 100
    
    collect_baseline_mean_fr_no_bursts.append(estimated_peak_baseline)
    collect_baseline_smooth_fr_no_bursts.append(np.mean(smooth_fr[bin_centers < 2]))

    collect_response_peak_mean_fr_no_bursts.append(estimated_peak_response)
    collect_response_peak_smooth_fr_no_bursts.append(max(smooth_fr))

    collect_response_period_mean_fr_no_bursts.append(np.mean(mean_fr_response) - gt_response_period_mean)
    collect_response_period_smooth_fr_no_bursts.append(np.mean(smooth_fr_response) - gt_response_period_mean)

    collect_isi_v_no_bursts.append(isis_v)
    
    n += 1

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

# Histograms
ax.hist(collect_baseline_mean_fr,
        bins=10,
        alpha=0.6,
        color="green",
        label="baseline mean, with bursts")

ax.hist(collect_baseline_mean_fr_no_bursts,
        bins=10,
        alpha=0.6,
        color="pink",
        label="baseline mean, without bursts")

# Get y-limits AFTER plotting histograms
ymin, ymax = ax.get_ylim()

# Ground-truth vertical line
ax.vlines(baseline_fr, ymin, ymax, colors="black", linestyles="dashed")

# Labels and styling
ax.set_xlabel("FR [Hz]")
ax.set_ylabel("Freq.")
ax.set_title(f"baseline mean - instant. firing rate\n{n_samples} simulations\nGT={baseline_fr}")

ax.legend()
sns.despine(ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

# Histograms
ax.hist(collect_baseline_smooth_fr,
        bins=10,
        alpha=0.6,
        color="green",
        label="baseline mean, with bursts")

ax.hist(collect_baseline_smooth_fr_no_bursts,
        bins=10,
        alpha=0.6,
        color="pink",
        label="baseline mean, without bursts")

# Get y-limits AFTER plotting histograms
ymin, ymax = ax.get_ylim()

# Ground-truth vertical line
ax.vlines(baseline_fr, ymin, ymax, colors="black", linestyles="dashed")

# Labels and styling
ax.set_xlabel("FR [Hz]")
ax.set_ylabel("Freq.")
ax.set_title(f"baseline mean - smoothed firing rate\n{n_samples} simulations\nGT={baseline_fr}")

ax.legend()
sns.despine(ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

# Histograms
ax.hist(collect_response_peak_mean_fr,
        bins=10,
        alpha=0.6,
        color="green",
        label="response mean, with bursts")

ax.hist(collect_response_peak_mean_fr_no_bursts,
        bins=10,
        alpha=0.6,
        color="pink",
        label="response mean, without bursts")

# Get y-limits AFTER plotting histograms
ymin, ymax = ax.get_ylim()

# Ground-truth vertical line
ax.vlines(response_fr, ymin, ymax, colors="black", linestyles="dashed")

# Labels and styling
ax.set_xlabel("FR [Hz]")
ax.set_ylabel("Freq.")
ax.set_title(f"response peak - instant. firing rate\n{n_samples} simulations\nGT={response_fr}")

ax.legend()
sns.despine(ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

# Histograms
ax.hist(collect_response_peak_smooth_fr,
        bins=10,
        alpha=0.6,
        color="green",
        label="response peak, with bursts")

ax.hist(collect_response_peak_smooth_fr_no_bursts,
        bins=10,
        alpha=0.6,
        color="pink",
        label="response peak, without bursts")

# Get y-limits AFTER plotting histograms
ymin, ymax = ax.get_ylim()

# Ground-truth vertical line
ax.vlines(response_fr, ymin, ymax, colors="black", linestyles="dashed")

# Labels and styling
ax.set_xlabel("FR [Hz]")
ax.set_ylabel("Freq.")
ax.set_title(f"response peak - smoothed firing rate\n{n_samples} simulations\nGT={response_fr}")

ax.legend()
sns.despine(ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

# Histograms
ax.hist(collect_response_period_smooth_fr,
        bins=10,
        alpha=0.6,
        color="green",
        label="response peak, with bursts")

ax.hist(collect_response_period_smooth_fr_no_bursts,
        bins=10,
        alpha=0.6,
        color="pink",
        label="response peak, without bursts")

# Get y-limits AFTER plotting histograms
ymin, ymax = ax.get_ylim()

# Ground-truth vertical line
ax.vlines(0, ymin, ymax, colors="black", linestyles="dashed")

# Labels and styling
ax.set_xlabel("FR [Hz]")
ax.set_ylabel("Freq.")
ax.set_title(f"difference in response period mean - smoothed firing rate\n{n_samples} simulations\nGT={round(gt_response_period_mean, 2)}")

ax.legend()
sns.despine(ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

# Histograms
ax.hist(collect_isi_v,
        bins=10,
        alpha=0.6,
        color="green",
        label="ISI violations, with bursts")

ax.hist(collect_isi_v_no_bursts,
        bins=10,
        alpha=0.6,
        color="pink",
        label="ISI violations, without bursts")

# Get y-limits AFTER plotting histograms
ymin, ymax = ax.get_ylim()

# Ground-truth vertical line
ax.vlines(0, ymin, ymax, colors="black", linestyles="dashed")

# Labels and styling
ax.set_xlabel("% ISIs < 3 ms")
ax.set_ylabel("Freq.")
ax.set_title(f"% ISI violations\n{n_samples} simulations")

ax.legend()
sns.despine(ax=ax)

plt.tight_layout()
plt.show()